<a href="https://colab.research.google.com/github/Zekeriya-Ui/main/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis**: One row represents the aggregated performance metrics for a unique `content_id` (a visible page or content item).

**Time Window**: The data is partitioned into an earlier window for training the model and a later, distinct window for holdout/scoring, reflecting daily performance data. In this sample, a random split simulates this time-based division.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label**: `residual_model` (Opportunity Score) - Calculated as `actual CTR - expected CTR` from the advanced model. This is the primary target for ranking content.

**Features (used in the advanced model)**:
- `avg_position`: Average search position of the content.
- `log_impressions`: Log-transformed total impressions (`np.log1p(total_impressions)`), to account for visibility.
- `n_queries`: Number of distinct queries associated with the content (query diversity).
- `content_type`: Categorical feature representing the type of content (one-hot encoded for the model).

**Context (identifiers or derived metrics not directly used as features but important for analysis/interpretation)**:
- `content_id`: Unique identifier for each content item.
- `total_impressions`: Sum of impressions for the content.
- `total_clicks`: Sum of clicks for the content.
- `ctr`: Click-through rate (`total_clicks / total_impressions`).
- `pos_bucket`: Rounded average position, used for baseline model.
- `expected_ctr_baseline`: Expected CTR based on the baseline model.
- `expected_ctr_model`: Expected CTR based on the advanced model.
- `word_count`: Number of words in the content (available from `dim_content`, but not included in `feats` for the final model).

**Excluded (from initial raw data)**:
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`: These raw granular metrics are aggregated and transformed into `total_impressions`, `total_clicks`, and `avg_position` at the `content_id` level. The raw, daily-level metrics are not used directly in the aggregated models.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This section verifies the data processing steps, ensuring that the claims made in Section 1 and 2 are supported by the data.

- **Grain**: Confirms that after aggregation, each row uniquely represents a `content_id`.
- **Counts**: Shows the number of records at various stages (raw, train, holdout, aggregated).
- **Missing Values**: Checks for any missing values in the key features used by the model.
- **Windows**: Verifies the split of data into training and holdout sets.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Sample Data Limitation**: This analysis is performed on a sample Parquet file (`fact_content_daily_performance_sample.parquet`) and not the full production dataset. Therefore, the findings and derived opportunity scores are representative of the sample and may not fully generalize to the entire corpus of content or across different time periods.

- **Simulated Time Split**: For demonstration purposes, the time-aware split between training and holdout windows is simulated using a random row-level split (`frac=0.7`). In a real-world scenario, distinct chronological periods would be explicitly used to prevent data leakage and accurately assess model stability over time. This simulation limits the ability to truly evaluate the *stability check* mentioned in the methodology.

- **GSC-only data**: The `FACT_PERFORMANCE` table focuses on Google Search Console (GSC) metrics. This means the model does not account for traffic or engagement from other sources (e.g., social media, direct traffic, other search engines), which could also contribute to content performance.

- **Implicit Time-Series**: While the methodology emphasizes time-aware splits, the specific time granularity and continuity of the `FACT_PERFORMANCE` data are not explicitly checked for gaps or irregular reporting, which could impact time-series modeling if implemented.

- **Exclusion of `word_count` from Model Features**: Although `word_count` is available in `DIM_CONTENT`, it is not included in the final `feats` list for the Ridge model. This limits the model's ability to incorporate content length as a direct predictor of CTR beyond its potential correlation with other features.

- **Proxy for Opportunity**: The `residual_model` is a proxy for "opportunity." It identifies underperforming pages *relative to their peers at the same position and with similar covariates*. It doesn't necessarily indicate *why* a page is underperforming or guarantee that an optimization will improve its CTR. Further qualitative analysis or A/B testing would be needed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.